In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd 
import plotly.express as px 
import plotly.io as pio
pio.renderers.default = "iframe"

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Reading Files

events = pd.read_csv('/kaggle/input/social-media-advertisement-performance/ad_events.csv')
ads = pd.read_csv('/kaggle/input/social-media-advertisement-performance/ads.csv')
campaigns = pd.read_csv('/kaggle/input/social-media-advertisement-performance/campaigns.csv')
users = pd.read_csv('/kaggle/input/social-media-advertisement-performance/users.csv')

In [ ]:
events.head()

In [ ]:
ads.head()

In [ ]:
campaigns.head()

In [ ]:
users.head()

#  Users Data Check

In [ ]:
users.info()

In [ ]:
users.isnull().sum()

# Handling duplicates  In "Users"


In [ ]:
users.duplicated("user_id").sum()


In [ ]:
users.drop_duplicates(subset="user_id", inplace=True)

In [ ]:
users.duplicated("user_id").sum()

In [ ]:
# Spliting interests Column
users[["interest_1", "interest_2","interest_3"]] = (
    users["interests"]
    .str.split(",", expand=True)
    .apply(lambda x: x.str.strip())
)
users.drop("interests",axis=1,inplace=True)

# Filling nulls
users.fillna({
    "interest_1": "Unknown",
    "interest_2": "Unknown",
    "interest_3":"Unknown"
}, inplace=True)
users.info()

In [ ]:
users.head()

# ADs Data Check

In [ ]:
ads.info()

In [ ]:
ads.duplicated("ad_id").sum()

In [ ]:
# Converting types of IDs
ads["ad_id"] = ads["ad_id"].astype(str)
ads['campaign_id'] = ads['campaign_id'].astype(str)
ads.info()

In [ ]:
# Spliting interests
ads[["target_interest_1", "target_interest_2"]] = (
    ads["target_interests"]
    .str.split(",", expand=True)
    .apply(lambda x: x.str.strip())
)
ads.drop("target_interests", axis=1 , inplace=True)

# Filling Nulls
ads
ads.info()


In [ ]:
ads.head()

# campaigns Data Check

In [ ]:
campaigns.info()

In [ ]:
# Converting ID into STR
campaigns["campaign_id"]=campaigns["campaign_id"].astype(str)
campaigns.info()

In [ ]:
# check nulls
campaigns.isnull().sum()

In [ ]:
campaigns.duplicated("campaign_id").sum()

In [ ]:
# converting start_date & end_date data types
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"]).dt.normalize()
campaigns["end_date"]= pd.to_datetime(campaigns["end_date"]).dt.normalize()
campaigns.info()

# AD Events Data Check

In [ ]:
events.info()

In [ ]:
events.duplicated("event_id").sum()

In [ ]:
events=events.drop_duplicates(subset="event_id")
events.duplicated("event_id").sum()

In [ ]:
# Converting IDs Data type
events["event_id"] = events["event_id"].astype(str)
events["ad_id"] = events["ad_id"].astype(str)
events.info()

In [ ]:
# Check for nulls
events.isnull().sum()

In [ ]:
events["timestamp"]=pd.to_datetime(events["timestamp"])
events.info()

In [ ]:
events.describe()

In [ ]:
ads.describe()

In [ ]:
campaigns.describe()

In [ ]:
users.describe()

# Data Merging

In [ ]:
df = events.merge(users, on="user_id") \
           .merge(ads, on="ad_id") \
           .merge(campaigns, on="campaign_id")
df.info()

### DataFrame `df` Summary

The merged DataFrame `df` contains **400,000 entries** and **28 columns**, providing a comprehensive view of ad events, user demographics, ad characteristics, and campaign details. Below is a summary of its structure:

**Dimensions:**
*   **Rows:** 400,000
*   **Columns:** 28

**Key Columns and Data Types:**

*   **Identifiers (object):** `event_id`, `ad_id`, `user_id`, `campaign_id` are stored as objects, indicating they are likely strings.
*   **Timestamps (datetime64[ns]):** `timestamp`, `start_date`, and `end_date` are correctly parsed as datetime objects, which is crucial for time-series analysis.
*   **Categorical Features (object):** `day_of_week`, `time_of_day`, `event_type`, `user_gender`, `age_group`, `country`, `location`, `interest_1`, `interest_2`, `interest_3`, `ad_platform`, `ad_type`, `target_gender`, `target_age_group`, `target_interest_1`, `target_interest_2`, and `name` are all objects, representing categorical data.
*   **Numerical Features:**
    *   `user_age` and `duration_days` are integers (`int64`).
    *   `total_budget` is a floating-point number (`float64`).

**Non-Null Counts:**
Almost all columns have 400,000 non-null entries, indicating a clean dataset with minimal missing values after the initial data cleaning steps. The only exception is `target_interest_2`, which has 199,833 non-null values, suggesting approximately half of the ads have a second target interest specified.

In [ ]:
df.head()

In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(f"\nColumn: {col}")
    print(f"Unique count: {df[col].nunique()}")
    print(df[col].unique()[:])

# EDA

In [ ]:
# User Age Distribution
fig = px.histogram(
    df,
    x="user_age",
    nbins=10,
    title="User Age Distribution"
)
fig.show()

In [ ]:
# Total Budget
fig_1= px.histogram(
    df,
    x="total_budget",
    nbins=10,
    title="Total Budget Distribution"
)
fig_1.show()

In [ ]:
# Duration Days Distribution
fig_2 = px.histogram(
    df,
     x="duration_days",
    nbins=10,
    title="Campaign Duration Distribution"
)
fig_2.show()

In [ ]:
# Outlier Check
fig = px.box(
    df,
    y="total_budget",
    title="Total Budget Boxplot"
)
fig.show()

In [ ]:
event_counts = (
    df["event_type"]
    .value_counts()
    .reset_index(name="count")
)

fig_3 = px.bar(
    event_counts,
    x="event_type",
    y="count",
    color="event_type",
    title="Event Type Distribution"
)

fig_3.show()

In [ ]:
fig_4 = px.pie(
    df,
    names="user_gender",
    title="User Gender Distribution"
)
fig_4.show()

In [ ]:
country_counts = df["country"].value_counts().reset_index(name="count")

fig_5 = px.bar(
    country_counts,
    x = "country",
    y ="count",
    title="ACtions Per Countey"
)
fig_5.show()

In [ ]:
df["date"] = df["timestamp"].dt.date

daily_events = df.groupby("date").size().reset_index(name="count")

fig_6 = px.line(
    daily_events,
    x="date",
    y="count",
    title="Daily Events Over Time"
)
fig_6.show()

In [ ]:
fig_7 = px.histogram(
    df,
    x="event_type",
    color="user_gender",
    barmode="group",
    title="Event Type by Gender"
)
fig_7.show()

In [ ]:
fig_8 = px.scatter(
    df,
    x="duration_days",
    y="total_budget",
    color="ad_platform",
    title="Budget vs Campaign Duration"
)
fig_8.show()

In [ ]:
fig_9 = px.box(
    df,
    x="event_type",
    y="user_age",
    color= "event_type",
    title="User Age by Event Type"
)
fig_9.show()

In [ ]:
# Compute normalized counts (percentage)
conversion = df["event_type"].value_counts(normalize=True).reset_index(name="percentage")

# Rename columns for clarity
conversion.columns = ["event_type", "percentage"]

# Plotly bar chart
fig_10 = px.bar(
    conversion,
    x="event_type",
    y="percentage",
    title="Event Type Percentage",
    labels={"event_type": "Event Type", "percentage": "Percentage"}
)

fig_10.show()

In [ ]:
# Count events per campaign
campaign_stats = (
    df.groupby(['campaign_id', 'ad_platform', 'name', 'total_budget'])
      .size()
      .reset_index(name='events_count')
)

campaign_stats.head(10)

In [ ]:
fig_budget_events = px.scatter(
    campaign_stats,
    x='total_budget',
    y='events_count',
    color='ad_platform',
    hover_data=['campaign_id', 'name'],
    size='events_count',  # optional: size proportional to number of events
    title='Campaign Performance: Total Budget vs Events Generated'
)
fig_budget_events.show()

In [ ]:
platform_stats = df.groupby('ad_platform').size().reset_index(name='total_events')

fig_platform = px.bar(
    platform_stats,
    x='ad_platform',
    y='total_events',
    color='ad_platform',
    text='total_events',
    title='Total Events by Ad Platform'
)
fig_platform.update_traces(textposition='outside')
fig_platform.show()

In [ ]:
fig_platform_budget = px.box(
    campaign_stats,
    x='ad_platform',
    y='total_budget',
    points='all',  # show individual campaigns
    title='Campaign Budgets by Platform'
)
fig_platform_budget.show()

In [ ]:
campaign_perf = (
    df.groupby(["campaign_id", "ad_platform"])
      .agg(
          impressions=("event_type", lambda x: (x == "Impression").sum()),
          clicks=("event_type", lambda x: (x == "Click").sum()),
          purchases=("event_type", lambda x: (x == "Purchase").sum()),
          total_budget=("total_budget", "first")  # assumes same budget per campaign
      )
      .reset_index()
)

campaign_perf["ctr"] = (
    campaign_perf["clicks"] /
    campaign_perf["impressions"].replace(0, 1)
)

campaign_perf["conversion_rate"] = (
    campaign_perf["purchases"] /
    campaign_perf["clicks"].replace(0, 1)
)

campaign_perf["cpa"] = (
    campaign_perf["total_budget"] /
    campaign_perf["purchases"].replace(0, 1)
)

# =========================
# Budget vs CTR
# =========================

fig_ctr = px.scatter(
    campaign_perf,
    x="total_budget",
    y="ctr",
    color="ad_platform",
    size="impressions",
    title="Budget vs CTR"
)

fig_ctr.show()


# =========================
# Budget vs Conversion Rate
# =========================

fig_cvr = px.scatter(
    campaign_perf,
    x="total_budget",
    y="conversion_rate",
    color="ad_platform",
    size="purchases",
    title="Budget vs Conversion Rate"
)

fig_cvr.show()


# =========================
# Budget vs Purchases
# =========================

fig_purchases = px.scatter(
    campaign_perf,
    x="total_budget",
    y="purchases",
    color="ad_platform",
    title="Budget vs Purchases"
)

fig_purchases.show()


In [ ]:
df["target_interest_2"] = df["target_interest_2"].fillna("Unknown")

all_targeted_interests = pd.concat([
    df["target_interest_1"],
    df["target_interest_2"]
])

interest_distribution = all_targeted_interests.value_counts().reset_index(name="count")
interest_distribution.columns = ["interest", "count"]

In [ ]:
fig_targeted_interests = px.bar(
    interest_distribution,
    x="interest",
    y="count",
    title="Distribution of Targeted Interests",
    labels={
        "interest": "Targeted Interest",
        "count": "Number of Ads"
    }
)
fig_targeted_interests.show()

In [ ]:
all_user_interests = pd.concat([
    df["interest_1"],
    df["interest_2"],
    df["interest_3"]
])

user_interest_distribution = all_user_interests.value_counts().reset_index(name="count")
user_interest_distribution.columns = ["interest", "count"]
user_interest_distribution.head()

In [ ]:
fig_user_interests = px.bar(
    user_interest_distribution,
    x="interest",
    y="count",
    title="Distribution of User Interests",
    labels={
        "interest": "User Interest",
        "count": "Number of Users"
    }
)
fig_user_interests.show()

In [ ]:
ad_type_distribution = df["ad_type"].value_counts().reset_index(name="count")
ad_type_distribution.columns = ["ad_type", "count"]
ad_type_distribution.head()

In [ ]:
fig_ad_type = px.bar(
    ad_type_distribution,
    x="ad_type",
    y="count",
    title="Distribution of Ad Types",
    labels={
        "ad_type": "Ad Type",
        "count": "Number of Ads"
    }
)
fig_ad_type.show()

In [ ]:
ad_type_performance = df.groupby('ad_type')['event_type'].value_counts().unstack(fill_value=0).reset_index()
ad_type_performance = ad_type_performance[['ad_type', 'Click', 'Purchase']]
ad_type_performance.head()

In [ ]:
fig_clicks_by_ad_type = px.bar(
    ad_type_performance,
    x='ad_type',
    y='Click',
    title='Clicks by Ad Type',
    labels={'ad_type': 'Ad Type', 'Click': 'Number of Clicks'}
)
fig_clicks_by_ad_type.show()

In [ ]:
fig_purchases_by_ad_type = px.bar(
    ad_type_performance,
    x='ad_type',
    y='Purchase',
    title='Purchases by Ad Type',
    labels={'ad_type': 'Ad Type', 'Purchase': 'Number of Purchases'}
)
fig_purchases_by_ad_type.show()

# **📊 Marketing Data Insights — Summary**
**🎯 1. Funnel Performance**

    - Ads achieve very high reach (~ 85% impressions), but conversion to purchases is very low (~0.5%).

    - There is a clear drop-off between clicks and purchases → the conversion funnel likely needs optimization.

**👥 2. Audience Demographics**

    - Most users are young (mainly 18–34 age group).

    - Gender distribution is balanced with similar engagement patterns across genders.

    - The highest activity comes from key markets such as United States, United Kingdom, Canada, and India.

**💰 3. Campaign Performance**

    - Campaign durations typically range from ~30–90 days with widely varying budgets.

    - Higher budgets increase reach but don’t consistently improve CTR, conversion rate, or purchases.

    - Budget alone is not a reliable predictor of campaign success.

**📱 4. Platform Performance**

    - Facebook generates more total events overall.

    - Instagram sometimes shows competitive conversion efficiency despite lower volume.

    - Platform effectiveness appears campaign-dependent rather than universally consistent.

**❤️ 5. Interests (Targeted vs. Actual Users)**

    - “Unknown” interests appear frequently, indicating incomplete targeting data.

    - Common targeted interests include health, finance, and travel.

    - Users often show interests in fitness, technology, lifestyle, and travel.

**📢 6. Ad Type Performance**

    - Stories ads are the most common and generate the most clicks and purchases.

    - Image and carousel ads perform moderately well.

    - Video ads are least frequent and generate fewer purchases, possibly due to higher production costs or niche usage.